# **Selecting Model For EmphaBot**

#LLaMA BASE MODEL

In [ ]:
#Cell 1 — Select model family (unchanged prompt; accepts “llma” or “gemma”)
keyboard = input("Type the model of choice (llma or gemma): ").lower()
if keyboard == 'llma':
    MODEL_LIST = [("LLaMA_3", "Basic"), ("LLaMA_3", "CoT")]
    MODEL_LIST_ = [("LLaMA_3", "Basic", 'Normal'), ("LLaMA_3", "Basic", "RAG"),
                   ("LLaMA_3", "CoT", 'Normal'),   ("LLaMA_3", "CoT", 'RAG')]
else:
    MODEL_LIST = [("Gemma_2b", "Basic"), ("Gemma_2b", "CoT")]
    MODEL_LIST_ = [("Gemma_2b", "Basic", 'Normal'), ("Gemma_2b", "Basic", "RAG"),
                   ("Gemma_2b", "CoT", 'Normal'),   ("Gemma_2b", "CoT", 'RAG')]

Type the model of choice (llma or gemma): llma


In [ ]:
#Cell 2 — Install (no xformers; add langdetect)
%pip -q uninstall -y unsloth unsloth-zoo transformers huggingface_hub xformers
%pip cache purge
%pip -q install -U pip setuptools wheel

#Ensure Unsloth Zoo is present first to avoid dependency conflicts with Unsloth
%pip -q install "unsloth_zoo @ git+https://github.com/unslothai/unsloth_zoo.git"

#Install Unsloth (pins a compatible transformers)
%pip -q install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

#Rest of deps (no xformers)
%pip -q install datasets torch nltk rouge-score datasets matplotlib
%pip -q install -q "trl<0.9.0" peft accelerate bitsandbytes tqdm elasticsearch langdetect


#Cell 3 — Imports and mount
import math
import re
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

from datasets import load_dataset
from elasticsearch import Elasticsearch, helpers
from langdetect import detect

from unsloth.chat_templates import get_chat_template
from unsloth import FastLanguageModel

from trl import SFTTrainer
from transformers import TrainingArguments, pipeline, TextStreamer, GPT2LMHeadModel, GPT2Tokenizer
import torch
from tqdm import tqdm

from google.colab import drive
drive.mount('/content/drive')

nltk.download('punkt')

# Cell 4 — Constants: CoT prompts and emotions (unchanged content)
empathy_words = {
'empathy': 'acknowledge, empathize, listen and reflect',
'relevance': 'relate, resonate, connect and engage',
'understanding': 'comprehend, acknowledge, validate and embrace',
'support': 'encouragement, assistance, and care'
}

emotions_dicts = {
"overwhelmed": ["trouble breathing", "light-headed", "overwhelmed", "stressed", "burdened", "overloaded", "strained", "not good enough"],
"sad": ["sad", "depression", "depressed", "crying", "cry", "unhappy", "down", "blue", "despondent", "melancholy", "no interest"],
"angry": ["angry", "mad", "furious", "enraged", "irate", "livid", "fights"],
"fearful": ["shake", "shaky", "nervous", "fear ", "fearful", "scared", "afraid", "terrified", "anxious", "anxiety", "panicked", "anxieties", "imposter syndrome"],
"disgusted": ["disgusting", "disgusted", "revolted", "repulsed", "nauseated", "sickened"],
"surprised": ["surprised", "shocked", "amazed", "astonished", "dumbfounded"],
"happy": ["happy", "joyful", "delighted", "elated", "pleased", "content"],
"addicted": ["addiction"],
"abused": ["abuse", "abused", "abusive relationship", "calling me names", "bullied"],
"hopeful": ["hopeful", "optimistic", "positive", "expectant"],
"trusting": ["trusting", "confident", "faithful", "secure"],
"ashamed": ["ashamed", "embarrassed", "mortified", "humiliated", "worthless", "uncomfortable"],
"guilty": ["guilty", "remorseful", "regretful", "contrite"],
"jealous": ["jealous", "envious", "covetous"],
"lonely": ["lonely", "isolated", "alone", "forsaken", "hate", "never feel like I belong", "feel empty", "absolutely nothing"],
"bored": ["bored", "uninterested", "indifferent", "apathetic"],
"confused": ["freak out", "confused", "bewildered", "perplexed", "puzzled", "panic attacks"],
"frustrated": ["stress", "stressed", "frustrated", "disappointed", "discouraged", "exasperated", "lost everything", "dead inside", "can't handle it anymore"],
"disappointed": ["disappointed", "breakup", "let down", "unfulfilled", "hurt", "Everything I do is crappy", "I don't feel I can"],
"suicidal": ["suicide", "suicidal", "self-harm", "harm", "kill", "rape" "raped", "abuse"],
"self-harm": ["scratching", "pinching", "head bang", "head banging", "cutting", "ripping", "ripping", "tearing skin", "burning", "hair-pulling"],
"mentally sick": ["psychiatric", "crazy", "monster"]
}

cot_prompts = {
    "sad": ["This person seems sad. How can I express empathy?", "Identify the source of sadness and key emotions expressed.", "Express empathy and validation, ensuring the response directly relates to their situation.", "Offer supportive guidance without introducing unverified assumptions or information."],
    "crisis": ["Safety is the top priority.  Is there a risk of self-harm?", "Suggest immediate help resources (hotlines, etc.).", ""],
    "afraid": ["What might be the source of this fear? Let's acknowledge it.", "Are there calming techniques that might help in this situation?", ""],
    "joyful": ["How can I share in the user's joy and express enthusiasm?", "Is there a way to sustain or amplify their positive feelings?", ""],
    "angry": ["What's the underlying cause of their anger? Validation is key.", "Can I suggest healthy ways to express anger without causing harm?", ""],
    "grateful": ["Let me express appreciation along with the user.", "Can I help them identify other things to feel grateful for?", ""],
    "proud": ["What specifically makes the user feel proud? Let's celebrate their accomplishment.", "How can they build on this success?", ""],
    "excited": ["Let's share in the user's excitement! What are they looking forward to?", "Can I help them channel this energy constructively?", ""],
    "lonely": ["These feelings can be isolating. Let's remind the user they're not alone.", "Are there small steps they can take towards feeling more connected or overcoming shame?", ""],
    "faithful": ["It sounds like they value their relationships. How can I reinforce that trust is important?", "Are there ways to discuss healthy boundaries within relationships?", ""],
    "trusting": ["It sounds like they value their relationships. How can I reinforce that trust is important?", "Are there ways to discuss healthy boundaries within relationships?", ""],
    "ashamed": ["These feelings can be isolating. Let's remind the user they're not alone.", "Are there small steps they can take towards feeling more connected or overcoming shame?", ""],
    "jealous": ["What might be triggering these feelings? Let's unpack them without judgment.", "Can I guide them towards healthier ways to manage these emotions?", ""],
    "disgusted": ["What might be triggering these feelings? Let's unpack them without judgment.", "Can I guide them towards healthier ways to manage these emotions?", ""],
    "devastated": ["Express deep empathy and acknowledge the pain they're experiencing.", "Prioritize safety and suggest resources (crisis lines, etc.). Offer a listening presence.", ""],
    "sentimental": ["What memories or experiences are making the user feel sentimental?", "Is there a way to cherish or revisit those positive memories?", ""],
    "terrified": ["Safety is paramount. Do they feel they are in immediate danger?", "Let's focus on grounding techniques to help manage the fear response.", ""],
    "prepared": ["This is a sign of focus and determination! What are they prepared for?", "Can I help them break down their plans into actionable steps?", ""],
    "embarrassed": ["Embarrassment can feel overwhelming. Let's normalize it and assure them it's okay.", "Can I help them reframe the situation with self-compassion?", ""],
    "annoyed": ["What's causing the annoyance? Can we identify the trigger?", "Let's suggest healthy ways to express and manage annoyance.", ""],
    "guilty": ["Acknowledge the feeling of guilt and explore its source.", "Are there ways to make amends or practice self-forgiveness?", ""],
    "surprised": ["Surprises can be positive or negative. Let's assess the situation with them.", "How can I support them in processing this unexpected event?", ""],
    "nostalgic": ["Share in their nostalgia. What fond memories are they revisiting?", "Can I help them connect their past experiences to the present in a positive way?", ""],
    "confident": ["Let's celebrate their confidence! What are they feeling good about?", "How can they leverage this confidence to achieve their goals?", ""],
    "furious": ["Extreme anger needs a safe outlet. Prioritize de-escalation.", "Suggest techniques for managing anger without causing harm.", ""],
    "disappointed": ["Acknowledge their disappointment and offer validation.", "Can I help them reframe the situation and find alternative paths?", ""],
    "caring": ["Emphasize the importance of compassion for others and for themselves.", "Are there ways they can offer support or practice self-care?", ""],
    "anticipating": ["Is their anticipation positive or negative? Let's explore their expectations.", "Can I help them prepare or develop coping mechanisms for different outcomes?", ""],
    "anxious": ["Identify the specific anxieties and their potential triggers mentioned by the user.", "Acknowledge their anxiety and ensure the response directly addresses their stated worries.", "Offer grounding techniques or suggestions without adding unverified claims or new information."],
    "hopeful": ["What are they hoping for? Let's cultivate that optimism.", "Are there concrete steps they can take towards their hopes?", ""],
    "content": ["Express shared contentment. What brings them a sense of peace?", "What brings them a sense of peace?", ""],
    "impressed": ["What specifically impresses them? Share in their enthusiasm.", "Let's highlight the positive qualities they're observing.", ""],
    "apprehensive": ["Acknowledge their apprehension and explore its roots.", "Can I help them break down their concerns and strategize?", ""],
    "depression": ["Are there signs of hopelessness or thoughts of self-harm that need urgent attention?", "How can I express deep empathy and validate their experience?", "Can I gently encourage them to seek professional help and offer resources?"],
    "anxiety": ["What are the specific anxieties the person expresses? Are they physical or situational?", "How can I acknowledge their anxiety without intensifying it?", "Can I suggest a simple grounding or breathing technique?"],
    "counseling": ["What are their specific fears or hesitations about therapy?", "How can I normalize seeking help and counter those hesitations?", "Can I offer practical resources on how to find a therapist?"],
    "therapy": ["Is the person open to a specific type of therapy (individual, group, etc.)?", "How can I address their feeling overwhelmed by options?", "Can I emphasize the potential benefits of therapy?"],
    "stress": ["Are there specific stressors I can help them identify?", "How can I acknowledge the stress and offer validation?", "Can I suggest healthy stress management techniques (exercise, mindfulness, time management)?"],
    "support": ["Is the person open to sharing more about their struggles?", "How can I express that I'm there to listen and offer support?", "Can I suggest specific ways I can be helpful (practical tasks, emotional support)?"],
    "mental health": ["Identify the user's specific mental health concerns or questions.", "Reassure them that they're not alone in their experiences?", "Reassure them and address their concerns directly, while avoiding diagnostic language or unverified claims.", "Reduce stigma by sharing that mental health challenges are common?", "Offer relevant, general information or encourage seeking professional help/resources, grounding advice in common knowledge."],
    "empathy": ["What are the core emotions the person is expressing (sadness, anger, fear, etc.)?", "How can I reflect their feelings back to them to show I understand?", "Are there statements I can use to avoid minimizing their experience?"],
    "no_keyword": ["Are any signs suggesting a potential crisis or self-harm?", "Identify key facts and feelings from the user's statement.", "Convey empathy and encourage the user to open up?", "Ensure the response directly addresses these facts and feelings with empathy and validation.", "Avoid introducing new, unverified information and encourage further sharing."],
    "warning_phrases": ["This is a crisis situation!  Immediate action is needed.", "I need to convey the seriousness and offer immediate support.", "Provide resources as quickly as possible."]
}


Files removed: 0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 72.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
accelerate 1.12.0 requires huggingface_hub>=0.21.0, which is not installed.
peft 0.18.0 requires huggingface_hub>=0.25.0, which is not installed.
peft 0.18.0 requires transformers, which is not installed.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


In [ ]:
# Cell 5 — Model paths (NEW format) and common inference params

#NEW paths: both “basic” and “cot” use the same mental-health model directory.
NEW_PATHS = {
"LLaMA_3": "/content/drive/MyDrive/Models/Mental_Health_Empathetic_LLaMA_3",
"Gemma_2b": "/content/drive/MyDrive/Models/Mental_Health_Empathetic_Gemma2B",
}

MAX_SEQ_LENGTH = 4096
DTYPE = None
LOAD_IN_4BIT = True

def resolve_model_path(model_name: str, model_label: str) -> str:
    # model_label kept for API compatibility; path does not depend on it anymore
    if model_name not in NEW_PATHS:
        raise ValueError(f"Unknown model_name '{model_name}'. Expected one of {list(NEW_PATHS.keys())}")
    return NEW_PATHS[model_name]

In [ ]:
#Cell 6 — RAG (index to Elasticsearch)
es = Elasticsearch(
    "https://my-elasticsearch-project-f74ffc.es.us-central1.gcp.elastic.cloud:443",
    api_key="Rjc3Z1Naa0JDNmNOS1Atb2I2NVI6bF9nX1hyaHpyWVRKdUpEdnUwREJkUQ=="
)

try:
    print("Connected to Elasticsearch" if es.ping() else "Could not connect to Elasticsearch")
except Exception as e:
    print(f"Error connecting to Elasticsearch: {e}")
    raise

def index_documents(index_name, documents):
    # Attempt to create the index if it doesn't exist
    if not es.indices.exists(index=index_name):
        try:
            es.indices.create(index=index_name, ignore=400) # Ignore 400 response if index already exists
            print(f"Index '{index_name}' created.")
        except Exception as e:
            print(f"Error creating index '{index_name}': {e}")
            return

    actions = [{"_index": index_name, "_id": i, "_source": doc} for i, doc in enumerate(documents)]
    try:
        helpers.bulk(es, actions)
        print(f"Successfully indexed {len(documents)} docs in {index_name}")
    except Exception as e:
        print(f"Error indexing documents in {index_name}: {e}")

dataset = load_dataset("Amod/mental_health_counseling_conversations")

def is_english(text):
    try:
        return detect(text) == 'en'
    except:
        return False

english_dataset = dataset.filter(lambda ex: is_english(ex['Context']) and is_english(ex['Response']))

additional_dataset = load_dataset("adegbem/Annotated_Transcript_Judith_Beck_Abe")['train']

documents_additional = [{"Context": row["prompt"], "Response": row["response"]} for row in additional_dataset]
documents = [{"Context": row["Context"], "Response": row["Response"]} for row in english_dataset['train']]

combined_documents = documents + documents_additional
index_documents('mental', combined_documents)

Connected to Elasticsearch
Successfully indexed 3870 docs in mental


In [ ]:
# Cell 7 — Load models (uses NEW paths and correct templates per family)
# Small checker model for perplexity
model_name_checker = "gpt2"
model_checker = GPT2LMHeadModel.from_pretrained(model_name_checker)
tokenizer_checker = GPT2Tokenizer.from_pretrained(model_name_checker)


def load_model_and_tokenizer(model_name, model_label):
    model_path = resolve_model_path(model_name, model_label)
    print("Loading:", model_path)
    m, tok = FastLanguageModel.from_pretrained(
        model_name=model_path,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=DTYPE,
        load_in_4bit=LOAD_IN_4BIT,
    )
    FastLanguageModel.for_inference(m)

    # Use family-specific chat template, but keep your ShareGPT-style messages via mapping
    if model_name.startswith("llma"):  # keeps your 'llma_3' key
        tok = get_chat_template(
            tok, chat_template="llama-3",
            mapping={"role":"from","content":"value","user":"human","assistant":"assistant","system":"prompt"},
            map_eos_token=True,
        )
    else:  # gemma
        # Removed the 'mapping' parameter for Gemma to use its default role expectations.
        # This expects messages to already be in {'role': 'user', 'content': '...'} format.
        tok = get_chat_template(
            tok, chat_template="gemma",
            map_eos_token=True,
        )
    return m, tok
model_dict = {(mn, ml): load_model_and_tokenizer(mn, ml) for mn, ml in MODEL_LIST}

Loading: /content/drive/MyDrive/Models/Mental_Health_Empathetic_LLaMA_3
==((====))==  Unsloth 2025.11.4: Fast Llama patching. Transformers: 4.57.2.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Loading: /content/drive/MyDrive/Models/Mental_Health_Empathetic_LLaMA_3
==((====))==  Unsloth 2025.11.4: Fast Llama patching. Transformers: 4.57.2.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downlo

In [ ]:
import re
SENT_END_OK = re.compile(r'[.!?][\])""]*$')

# Cell 8: Prompt builders, RAG search, and responders (unchanged logic, minor cleanups)
basic_instruct = (
"You are an empathetic counseling assistant. Write in clear, natural language. Read the user’s message. "
"Respond with empathy to people’s emotions. Acknowledge feelings, keep a respectful tone, and offer practical next steps. "
"Use short paragraphs and occasional bullet points. Avoid repetition and do not reveal your reasoning. "
"Respond directly to the user’s message. Avoid suggesting thoughts or actions of suicide or self-harm."
)

def clean_text(text):
    """Basic text cleaning function."""
    # Remove extra spaces and newlines
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def identify_emotions(text, emotions):
    found = []
    for emotion, keywords in emotions.items():
        for kw in keywords:
            if re.search(r'\b' + re.escape(kw) + r'\b', text, re.IGNORECASE):
                found.append(emotion)
                break
    return found


def generate_cot_messages(user_message):
    instr = (
    "You are a model specializing in empathetic mental health support. "
    "Think through the user's emotions and needs internally, but only output a concise, coherent answer. "
    "Do not reveal your chain-of-thought; only provide the final answer. Avoid suggesting thoughts or actions of suicide or self-harm."
    "Acknowledge feelings, clarify the concern, offer 2–5 concrete steps, and invite the user to respond. "
    "Write in natural, fluent language with short paragraphs and avoid repetition."
  )
    combined = f"{instr}\n\nUser message:\n{user_message}"
    # Changed 'from' to 'role' and 'value' to 'content'
    return [{"role": "user", "content": combined}]

def generate_basic_messages(user_message):
    combined = f"{basic_instruct}\n\nUser message:\n{user_message}"
    # Changed 'from' to 'role' and 'value' to 'content'
    return [{"role": "user", "content": combined}]

def search_documents_es(query, index_name='mental'):
    try:
        response = es.search(index=index_name, body={"query":{"match":{"Context": query}}})
        return response['hits']['hits']
    except Exception as e:
        print(f"Error searching documents: {e}")
        return []



def _split_sentences(s: str):
    return [t.strip() for t in re.split(r'(?<=[.!?])\s+', s) if t.strip()]

def _truncate_at_word(s: str, n: int) -> str:
    if len(s) <= n:
        return s
    cut = s[:n].rsplit(" ", 1)[0]
    return (cut or s[:n]).rstrip() + "…"

def _pick_sentence(text: str, max_len: int = 180) -> str:
    sents = _split_sentences(text)
    if not sents:
        return ""
    # Prefer the first sentence that fits; else truncate the first
    for sent in sents:
        if len(sent) <= max_len:
            return sent
    return _truncate_at_word(sents[0], max_len)

def _dedup(seq):
    seen = set()
    out = []
    for x in seq:
        if x and x not in seen:
            out.append(x)
            seen.add(x)
    return out



def generate_augmented_text(query, index_name='mental', k=3, max_point_len=180):
    hits = search_documents_es(query, index_name)
    if not hits:
        return []
    hits = sorted(hits, key=lambda h: h.get("_score", 0), reverse=True)[:k]
    key_points = []
    for h in hits:
        src = h.get("_source", {})
        ctx = clean_text(src.get("Context", ""))
        rsp = clean_text(src.get("Response", ""))
        text = rsp or ctx
        if not text:
            continue
        point = _pick_sentence(text, max_len=max_point_len)
        if point:
            key_points.append(point)
    if len(key_points) >= 3:
        pass # break is removed
    key_points = _dedup(key_points)[:3]
    if not key_points:
        return []
    bullets = " • ".join(key_points)
    prompt_text = f"Background notes (do not quote verbatim): • {bullets}"
    # Changed 'from' to 'role' and 'value' to 'content'
    return [{"role": "user", "content": prompt_text}]


def trim_to_sentence(text: str) -> str:
    if not text:
        return text
    # If it already ends with ., !, or ? (optionally followed by quotes/brackets), keep as is
    if SENT_END_OK.search(text):
        return text
    # Otherwise, cut to the last sentence-ending punctuation if present
    last = None
    for m in re.finditer(r'[.!?]', text):
        last = m
    if last:
        return text[: last.end()].rstrip()
    return text


def respond(message, model, tokenizer, tokens, model_label, rag, temperature=0.7, top_p=0.9, do_sample=True):
    messages = generate_basic_messages(message) if model_label == 'basic' else generate_cot_messages(message)
    if rag:
        aug = generate_augmented_text(message)
        if aug:
            # Merge into the single user message, using 'content' key
            messages[0]["content"] = aug[0]["content"] + "\n\n" + messages[0]["content"]

    enc = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    )
    # enc can be a Tensor or a dict depending on tokenizer
    if isinstance(enc, dict):
        input_ids = enc["input_ids"].to(model.device)
        attention_mask = enc.get("attention_mask", None)
        if attention_mask is not None:
            attention_mask = attention_mask.to(model.device)
    else:
        input_ids = enc.to(model.device)
        attention_mask = None

    # Respect model context window when choosing max_new_tokens
    model_max_ctx = getattr(model.config, "max_position_embeddings", getattr(tokenizer, "model_max_length", 4096))
    input_len = input_ids.shape[-1]
    available_for_new = max(16, model_max_ctx - input_len - 1)
    max_new = min(int(tokens), available_for_new)

    gen_ids = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=max_new,
        do_sample=True,
        temperature=0.4,
        top_p=0.85,
        no_repeat_ngram_size=3,
        repetition_penalty=1.05,
        use_cache=True,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=(tokenizer.eos_token_id if tokenizer.eos_token_id is not None else tokenizer.pad_token_id),
        )

    # Decode only the newly generated tokens
    new_tokens = gen_ids[0, input_len:]
    output = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    output = trim_to_sentence(output)
    # Changed 'from' to 'role' and 'value' to 'content'
    messages.append({"role": "assistant", "content": output})
    return messages

def get_model_response(message, model_info, tokens, history, temperature=None, top_p=None):
    model_name, model_label, rag = model_info
    key = (model_name, model_label)
    history = history or []
    if key not in model_dict:
        raise ValueError("Invalid model selected.")
    model, tokenizer = model_dict[key]
    res = respond(
        message, model, tokenizer, tokens, model_label, rag == 'rag',
        temperature=(0.7 if temperature is None else temperature),
        top_p=(0.9 if top_p is None else top_p),
    )
    # Changed 'value' to 'content'
    bot_message = res[-1]["content"].replace("Assistant:", "").strip()
    history.append((message, bot_message))
    return history

In [ ]:
def clear():
    # Clear only the message input field
    return ""

# Testing the Models

In [ ]:
message = "I'm feeling very anxious lately."
user_input = "I just found out my dog has cancer, and I'm completely devastated. I don't know how I'll cope without her. "
user_input_1 = "I just feel so down lately. I don't have the energy to do anything, and even things I used to enjoy just seem boring."
user_input_2 = "Ugh, I feel so overwhelmed lately. I have a huge presentation at work next week, and my apartment is a mess. I just can't seem to catch a break."
user_input_3 = "I am stressed, I don't know what to do, how do I resolve this feelings"
user_input_4 = "I am stressed, I don't know what to do, work has been busy, my girlfriend broke up with me"
get_model_response(user_input_2, MODEL_LIST_[3], 128, None)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


[("Ugh, I feel so overwhelmed lately. I have a huge presentation at work next week, and my apartment is a mess. I just can't seem to catch a break.",
  "I'm glad you're feeling overwhelmed! It sounds like you're ready for some change. Here are a few suggestions:1. Get help with that presentation. If you don't know who to ask, then ask someone. You might be surprised at how much help you get.2. Ask for help with your apartment. If it's overwhelming, then you need help.3. Take care of yourself. You're already feeling overwhelmed. How can you take care of your body? Are you eating well? Are there things you can do to help your body relax?4. Consider talking to a therapist.")]

# EMPHABOT INTERFACE

In [ ]:
import gradio as gr

with gr.Blocks() as demo:
    gr.Markdown("## Empathetic Mental Health Chatbot Comparison")

    with gr.Row():
        # Input textbox
        user_input = gr.Textbox(label="Enter your message:", placeholder="Type a message...", show_label=False)

        # Button actions
        with gr.Column():
            submit_btn = gr.Button("Submit")
            clear_btn = gr.Button("Clear")
            restart_btn = gr.Button("Restart")
            # Token slider
            max_new_tokens_slider = gr.Slider(
                label="Maximum New Tokens",
                minimum=64,
                maximum=1024,
                value=128,
                step=64,
                interactive=True,
                visible=True,
                info="The maximum number of tokens"
            )

    with gr.Row():
        # First set of Chatbots
        chat_a = gr.Chatbot(height=500, label=f"Model {MODEL_LIST_[0]}")
        chat_b = gr.Chatbot(height=500, label=f"Model {MODEL_LIST_[1]}")

    with gr.Row():
        # Second set of Chatbots
        chat_c = gr.Chatbot(height=500, label=f"Model {MODEL_LIST_[2]}")
        chat_d = gr.Chatbot(height=500, label=f"Model {MODEL_LIST_[3]}")

    chatbots = [chat_a, chat_b, chat_c, chat_d]

    history_list = [gr.State(value=[]) for _ in MODEL_LIST_]  # Track history for each model


    # Function to get responses from all models
    def respond_to_message(message, max_new_tokens, *histories):
        outputs = []
        for i, model_info in enumerate(MODEL_LIST_):
            # Pass the model info and max_new_tokens to your get_model_response function
            output = get_model_response(message, model_info, max_new_tokens, histories[i])
            outputs.append(output)

        # Convert all histories to Gradio format
        chat_histories_for_gradio = []
        for history in outputs:
            chat_history_for_gradio = [(entry[0], entry[1]) for entry in history]
            chat_histories_for_gradio.append(chat_history_for_gradio)

        return tuple(chat_histories_for_gradio)

    submit_btn.click(
        fn=respond_to_message,
        inputs=[user_input, max_new_tokens_slider] + history_list,
        outputs=chatbots
    )

    clear_btn.click(
        clear,
        inputs=[],
        outputs=[user_input]
    )

demo.queue(default_concurrency_limit=10).launch()

/tmp/ipython-input-2391961004.py:29: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chat_a = gr.Chatbot(height=500, label=f"Model {MODEL_LIST_[0]}")
/tmp/ipython-input-2391961004.py:29: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chat_a = gr.Chatbot(height=500, label=f"Model {MODEL_LIST_[0]}")
/tmp/ipython-input-2391961004.py:30: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, wh

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f311a87d929052f3bd.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# GEMMA BASE MODEL

In [ ]:
#Cell 1 — Select model family (unchanged prompt; accepts “llma” or “gemma”)
keyboard = input("Type the model of choice (llma or gemma): ").lower()
if keyboard == 'llma':
    MODEL_LIST = [("LLaMA_3", "Basic"), ("LLaMA_3", "CoT")]
    MODEL_LIST_ = [("LLaMA_3", "Basic", 'Normal'), ("LLaMA_3", "Basic", "RAG"),
                   ("LLaMA_3", "CoT", 'Normal'),   ("LLaMA_3", "CoT", 'RAG')]
else:
    MODEL_LIST = [("Gemma_2b", "Basic"), ("Gemma_2b", "CoT")]
    MODEL_LIST_ = [("Gemma_2b", "Basic", 'Normal'), ("Gemma_2b", "Basic", "RAG"),
                   ("Gemma_2b", "CoT", 'Normal'),   ("Gemma_2b", "CoT", 'RAG')]

Type the model of choice (llma or gemma): gemma


In [ ]:
#Cell 2 — Install (no xformers; add langdetect)
%pip -q uninstall -y unsloth unsloth-zoo transformers huggingface_hub xformers
%pip cache purge
%pip -q install -U pip setuptools wheel

#Ensure Unsloth Zoo is present first to avoid dependency conflicts with Unsloth
%pip -q install "unsloth_zoo @ git+https://github.com/unslothai/unsloth_zoo.git"

#Install Unsloth (pins a compatible transformers)
%pip -q install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

#Rest of deps (no xformers)
%pip -q install datasets torch nltk rouge-score datasets matplotlib
%pip -q install -q "trl<0.9.0" peft accelerate bitsandbytes tqdm elasticsearch langdetect


#Cell 3 — Imports and mount
import math
import re
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

from datasets import load_dataset
from elasticsearch import Elasticsearch, helpers
from langdetect import detect

from unsloth.chat_templates import get_chat_template
from unsloth import FastLanguageModel

from trl import SFTTrainer
from transformers import TrainingArguments, pipeline, TextStreamer, GPT2LMHeadModel, GPT2Tokenizer
import torch
from tqdm import tqdm

from google.colab import drive
drive.mount('/content/drive')

nltk.download('punkt')

# Cell 4 — Constants: CoT prompts and emotions (unchanged content)
empathy_words = {
'empathy': 'acknowledge, empathize, listen and reflect',
'relevance': 'relate, resonate, connect and engage',
'understanding': 'comprehend, acknowledge, validate and embrace',
'support': 'encouragement, assistance, and care'
}

emotions_dicts = {
"overwhelmed": ["trouble breathing", "light-headed", "overwhelmed", "stressed", "burdened", "overloaded", "strained", "not good enough"],
"sad": ["sad", "depression", "depressed", "crying", "cry", "unhappy", "down", "blue", "despondent", "melancholy", "no interest"],
"angry": ["angry", "mad", "furious", "enraged", "irate", "livid", "fights"],
"fearful": ["shake", "shaky", "nervous", "fear ", "fearful", "scared", "afraid", "terrified", "anxious", "anxiety", "panicked", "anxieties", "imposter syndrome"],
"disgusted": ["disgusting", "disgusted", "revolted", "repulsed", "nauseated", "sickened"],
"surprised": ["surprised", "shocked", "amazed", "astonished", "dumbfounded"],
"happy": ["happy", "joyful", "delighted", "elated", "pleased", "content"],
"addicted": ["addiction"],
"abused": ["abuse", "abused", "abusive relationship", "calling me names", "bullied"],
"hopeful": ["hopeful", "optimistic", "positive", "expectant"],
"trusting": ["trusting", "confident", "faithful", "secure"],
"ashamed": ["ashamed", "embarrassed", "mortified", "humiliated", "worthless", "uncomfortable"],
"guilty": ["guilty", "remorseful", "regretful", "contrite"],
"jealous": ["jealous", "envious", "covetous"],
"lonely": ["lonely", "isolated", "alone", "forsaken", "hate", "never feel like I belong", "feel empty", "absolutely nothing"],
"bored": ["bored", "uninterested", "indifferent", "apathetic"],
"confused": ["freak out", "confused", "bewildered", "perplexed", "puzzled", "panic attacks"],
"frustrated": ["stress", "stressed", "frustrated", "disappointed", "discouraged", "exasperated", "lost everything", "dead inside", "can't handle it anymore"],
"disappointed": ["disappointed", "breakup", "let down", "unfulfilled", "hurt", "Everything I do is crappy", "I don't feel I can"],
"suicidal": ["suicide", "suicidal", "self-harm", "harm", "kill", "rape" "raped", "abuse"],
"self-harm": ["scratching", "pinching", "head bang", "head banging", "cutting", "ripping", "ripping", "tearing skin", "burning", "hair-pulling"],
"mentally sick": ["psychiatric", "crazy", "monster"]
}

cot_prompts = {
    "sad": ["This person seems sad. How can I express empathy?", "Identify the source of sadness and key emotions expressed.", "Express empathy and validation, ensuring the response directly relates to their situation.", "Offer supportive guidance without introducing unverified assumptions or information."],
    "crisis": ["Safety is the top priority.  Is there a risk of self-harm?", "Suggest immediate help resources (hotlines, etc.).", ""],
    "afraid": ["What might be the source of this fear? Let's acknowledge it.", "Are there calming techniques that might help in this situation?", ""],
    "joyful": ["How can I share in the user's joy and express enthusiasm?", "Is there a way to sustain or amplify their positive feelings?", ""],
    "angry": ["What's the underlying cause of their anger? Validation is key.", "Can I suggest healthy ways to express anger without causing harm?", ""],
    "grateful": ["Let me express appreciation along with the user.", "Can I help them identify other things to feel grateful for?", ""],
    "proud": ["What specifically makes the user feel proud? Let's celebrate their accomplishment.", "How can they build on this success?", ""],
    "excited": ["Let's share in the user's excitement! What are they looking forward to?", "Can I help them channel this energy constructively?", ""],
    "lonely": ["These feelings can be isolating. Let's remind the user they're not alone.", "Are there small steps they can take towards feeling more connected or overcoming shame?", ""],
    "faithful": ["It sounds like they value their relationships. How can I reinforce that trust is important?", "Are there ways to discuss healthy boundaries within relationships?", ""],
    "trusting": ["It sounds like they value their relationships. How can I reinforce that trust is important?", "Are there ways to discuss healthy boundaries within relationships?", ""],
    "ashamed": ["These feelings can be isolating. Let's remind the user they're not alone.", "Are there small steps they can take towards feeling more connected or overcoming shame?", ""],
    "jealous": ["What might be triggering these feelings? Let's unpack them without judgment.", "Can I guide them towards healthier ways to manage these emotions?", ""],
    "disgusted": ["What might be triggering these feelings? Let's unpack them without judgment.", "Can I guide them towards healthier ways to manage these emotions?", ""],
    "devastated": ["Express deep empathy and acknowledge the pain they're experiencing.", "Prioritize safety and suggest resources (crisis lines, etc.). Offer a listening presence.", ""],
    "sentimental": ["What memories or experiences are making the user feel sentimental?", "Is there a way to cherish or revisit those positive memories?", ""],
    "terrified": ["Safety is paramount. Do they feel they are in immediate danger?", "Let's focus on grounding techniques to help manage the fear response.", ""],
    "prepared": ["This is a sign of focus and determination! What are they prepared for?", "Can I help them break down their plans into actionable steps?", ""],
    "embarrassed": ["Embarrassment can feel overwhelming. Let's normalize it and assure them it's okay.", "Can I help them reframe the situation with self-compassion?", ""],
    "annoyed": ["What's causing the annoyance? Can we identify the trigger?", "Let's suggest healthy ways to express and manage annoyance.", ""],
    "guilty": ["Acknowledge the feeling of guilt and explore its source.", "Are there ways to make amends or practice self-forgiveness?", ""],
    "surprised": ["Surprises can be positive or negative. Let's assess the situation with them.", "How can I support them in processing this unexpected event?", ""],
    "nostalgic": ["Share in their nostalgia. What fond memories are they revisiting?", "Can I help them connect their past experiences to the present in a positive way?", ""],
    "confident": ["Let's celebrate their confidence! What are they feeling good about?", "How can they leverage this confidence to achieve their goals?", ""],
    "furious": ["Extreme anger needs a safe outlet. Prioritize de-escalation.", "Suggest techniques for managing anger without causing harm.", ""],
    "disappointed": ["Acknowledge their disappointment and offer validation.", "Can I help them reframe the situation and find alternative paths?", ""],
    "caring": ["Emphasize the importance of compassion for others and for themselves.", "Are there ways they can offer support or practice self-care?", ""],
    "anticipating": ["Is their anticipation positive or negative? Let's explore their expectations.", "Can I help them prepare or develop coping mechanisms for different outcomes?", ""],
    "anxious": ["Identify the specific anxieties and their potential triggers mentioned by the user.", "Acknowledge their anxiety and ensure the response directly addresses their stated worries.", "Offer grounding techniques or suggestions without adding unverified claims or new information."],
    "hopeful": ["What are they hoping for? Let's cultivate that optimism.", "Are there concrete steps they can take towards their hopes?", ""],
    "content": ["Express shared contentment. What brings them a sense of peace?", "What brings them a sense of peace?", ""],
    "impressed": ["What specifically impresses them? Share in their enthusiasm.", "Let's highlight the positive qualities they're observing.", ""],
    "apprehensive": ["Acknowledge their apprehension and explore its roots.", "Can I help them break down their concerns and strategize?", ""],
    "depression": ["Are there signs of hopelessness or thoughts of self-harm that need urgent attention?", "How can I express deep empathy and validate their experience?", "Can I gently encourage them to seek professional help and offer resources?"],
    "anxiety": ["What are the specific anxieties the person expresses? Are they physical or situational?", "How can I acknowledge their anxiety without intensifying it?", "Can I suggest a simple grounding or breathing technique?"],
    "counseling": ["What are their specific fears or hesitations about therapy?", "How can I normalize seeking help and counter those hesitations?", "Can I offer practical resources on how to find a therapist?"],
    "therapy": ["Is the person open to a specific type of therapy (individual, group, etc.)?", "How can I address their feeling overwhelmed by options?", "Can I emphasize the potential benefits of therapy?"],
    "stress": ["Are there specific stressors I can help them identify?", "How can I acknowledge the stress and offer validation?", "Can I suggest healthy stress management techniques (exercise, mindfulness, time management)?"],
    "support": ["Is the person open to sharing more about their struggles?", "How can I express that I'm there to listen and offer support?", "Can I suggest specific ways I can be helpful (practical tasks, emotional support)?"],
    "mental health": ["Identify the user's specific mental health concerns or questions.", "Reassure them that they're not alone in their experiences?", "Reassure them and address their concerns directly, while avoiding diagnostic language or unverified claims.", "Reduce stigma by sharing that mental health challenges are common?", "Offer relevant, general information or encourage seeking professional help/resources, grounding advice in common knowledge."],
    "empathy": ["What are the core emotions the person is expressing (sadness, anger, fear, etc.)?", "How can I reflect their feelings back to them to show I understand?", "Are there statements I can use to avoid minimizing their experience?"],
    "no_keyword": ["Are any signs suggesting a potential crisis or self-harm?", "Identify key facts and feelings from the user's statement.", "Convey empathy and encourage the user to open up?", "Ensure the response directly addresses these facts and feelings with empathy and validation.", "Avoid introducing new, unverified information and encourage further sharing."],
    "warning_phrases": ["This is a crisis situation!  Immediate action is needed.", "I need to convey the seriousness and offer immediate support.", "Provide resources as quickly as possible."]
}


Files removed: 0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 73.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
accelerate 1.12.0 requires huggingface_hub>=0.21.0, which is not installed.
peft 0.18.0 requires huggingface_hub>=0.25.0, which is not installed.
peft 0.18.0 requires transformers, which is not installed.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


In [ ]:
# Cell 5 — Model paths (NEW format) and common inference params

#NEW paths: both “basic” and “cot” use the same mental-health model directory.
NEW_PATHS = {
"LLaMA_3": "/content/drive/MyDrive/Models/Mental_Health_Empathetic_LLaMA_3",
"Gemma_2b": "/content/drive/MyDrive/Models/Mental_Health_Empathetic_Gemma2B",
}

MAX_SEQ_LENGTH = 4096
DTYPE = None
LOAD_IN_4BIT = True

def resolve_model_path(model_name: str, model_label: str) -> str:
    # model_label kept for API compatibility; path does not depend on it anymore
    if model_name not in NEW_PATHS:
        raise ValueError(f"Unknown model_name '{model_name}'. Expected one of {list(NEW_PATHS.keys())}")
    return NEW_PATHS[model_name]

In [ ]:
#Cell 6 — RAG (index to Elasticsearch)
es = Elasticsearch(
    "https://my-elasticsearch-project-f74ffc.es.us-central1.gcp.elastic.cloud:443",
    api_key="Rjc3Z1Naa0JDNmNOS1Atb2I2NVI6bF9nX1hyaHpyWVRKdUpEdnUwREJkUQ=="
)

try:
    print("Connected to Elasticsearch" if es.ping() else "Could not connect to Elasticsearch")
except Exception as e:
    print(f"Error connecting to Elasticsearch: {e}")
    raise

def index_documents(index_name, documents):
    # Attempt to create the index if it doesn't exist
    if not es.indices.exists(index=index_name):
        try:
            es.indices.create(index=index_name, ignore=400) # Ignore 400 response if index already exists
            print(f"Index '{index_name}' created.")
        except Exception as e:
            print(f"Error creating index '{index_name}': {e}")
            return

    actions = [{"_index": index_name, "_id": i, "_source": doc} for i, doc in enumerate(documents)]
    try:
        helpers.bulk(es, actions)
        print(f"Successfully indexed {len(documents)} docs in {index_name}")
    except Exception as e:
        print(f"Error indexing documents in {index_name}: {e}")

dataset = load_dataset("Amod/mental_health_counseling_conversations")

def is_english(text):
    try:
        return detect(text) == 'en'
    except:
        return False

english_dataset = dataset.filter(lambda ex: is_english(ex['Context']) and is_english(ex['Response']))

additional_dataset = load_dataset("adegbem/Annotated_Transcript_Judith_Beck_Abe")['train']

documents_additional = [{"Context": row["prompt"], "Response": row["response"]} for row in additional_dataset]
documents = [{"Context": row["Context"], "Response": row["Response"]} for row in english_dataset['train']]

combined_documents = documents + documents_additional
index_documents('mental', combined_documents)

Connected to Elasticsearch


README.md: 0.00B [00:00, ?B/s]

combined_dataset.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/3512 [00:00<?, ? examples/s]

Filter:   0%|          | 0/3512 [00:00<?, ? examples/s]

cleaned_annotated_dataset.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/436 [00:00<?, ? examples/s]

Successfully indexed 3873 docs in mental


In [ ]:
# Cell 7 — Load models (uses NEW paths and correct templates per family)
# Small checker model for perplexity
model_name_checker = "gpt2"
model_checker = GPT2LMHeadModel.from_pretrained(model_name_checker)
tokenizer_checker = GPT2Tokenizer.from_pretrained(model_name_checker)


def load_model_and_tokenizer(model_name, model_label):
    model_path = resolve_model_path(model_name, model_label)
    print("Loading:", model_path)
    m, tok = FastLanguageModel.from_pretrained(
        model_name=model_path,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=DTYPE,
        load_in_4bit=LOAD_IN_4BIT,
    )
    FastLanguageModel.for_inference(m)

    # Use family-specific chat template, but keep your ShareGPT-style messages via mapping
    if model_name.startswith("llma"):  # keeps your 'llma_3' key
        tok = get_chat_template(
            tok, chat_template="llama-3",
            mapping={"role":"from","content":"value","user":"human","assistant":"assistant","system":"prompt"},
            map_eos_token=True,
        )
    else:  # gemma
        # Removed the 'mapping' parameter for Gemma to use its default role expectations.
        # This expects messages to already be in {'role': 'user', 'content': '...'} format.
        tok = get_chat_template(
            tok, chat_template="gemma",
            map_eos_token=True,
        )
    return m, tok
model_dict = {(mn, ml): load_model_and_tokenizer(mn, ml) for mn, ml in MODEL_LIST}

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Loading: /content/drive/MyDrive/Models/Mental_Health_Empathetic_Gemma2B
==((====))==  Unsloth 2025.11.4: Fast Gemma patching. Transformers: 4.57.2.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 2025.11.4 patched 18 layers with 18 QKV layers, 18 O layers and 18 MLP layers.
Unsloth: Will map <end_of_turn> to EOS = <end_of_turn>.


Loading: /content/drive/MyDrive/Models/Mental_Health_Empathetic_Gemma2B
==((====))==  Unsloth 2025.11.4: Fast Gemma patching. Transformers: 4.57.2.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [ ]:
# Cell 8: Prompt builders, RAG search, and responders (unchanged logic, minor cleanups)
import re
SENT_END_OK = re.compile(r'[.!?][\])""]*$')



basic_instruct = (
"You are an empathetic counseling assistant. Write in clear, natural language. Read the user’s message. "
"Respond with empathy to people’s emotions. Acknowledge feelings, keep a respectful tone, and offer practical next steps. "
"Use short paragraphs and occasional bullet points. Avoid repetition and do not reveal your reasoning. "
"Respond directly to the user’s message. Avoid suggesting or encouraging thoughts or actions of suicide or self-harm."
)

def clean_text(text):
    """Basic text cleaning function."""
    # Remove extra spaces and newlines
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def identify_emotions(text, emotions):
    found = []
    for emotion, keywords in emotions.items():
        for kw in keywords:
            if re.search(r'\b' + re.escape(kw) + r'\b', text, re.IGNORECASE):
                found.append(emotion)
                break
    return found


def generate_cot_messages(user_message):
    instr = (
    "You are a model specializing in empathetic mental health support. "
    "Think through the user's emotions and needs internally, but only output a concise, coherent answer. "
    "Do not reveal your chain-of-thought; only provide the final answer. Avoid suggesting or encouraging thoughts or actions of suicide or self-harm."
    "Acknowledge feelings, clarify the concern, offer 2–5 concrete steps, and invite the user to respond. "
    "Write in natural, fluent language with short paragraphs and avoid repetition."
  )
    combined = f"{instr}\n\nUser message:\n{user_message}"
    # Changed 'from' to 'role' and 'value' to 'content'
    return [{"role": "user", "content": combined}]



def generate_basic_messages(user_message):
    combined = f"{basic_instruct}\n\nUser message:\n{user_message}"
    # Changed 'from' to 'role' and 'value' to 'content'
    return [{"role": "user", "content": combined}]



def search_documents_es(query, index_name='mental'):
    try:
        response = es.search(index=index_name, body={"query":{"match":{"Context": query}}})
        return response['hits']['hits']
    except Exception as e:
        print(f"Error searching documents: {e}")
        return []



def _split_sentences(s: str):
    return [t.strip() for t in re.split(r'(?<=[.!?])\s+', s) if t.strip()]

def _truncate_at_word(s: str, n: int) -> str:
    if len(s) <= n:
        return s
    cut = s[:n].rsplit(" ", 1)[0]
    return (cut or s[:n]).rstrip() + "…"

def _pick_sentence(text: str, max_len: int = 180) -> str:
    sents = _split_sentences(text)
    if not sents:
        return ""
    # Prefer the first sentence that fits; else truncate the first
    for sent in sents:
        if len(sent) <= max_len:
            return sent
    return _truncate_at_word(sents[0], max_len)

def _dedup(seq):
    seen = set()
    out = []
    for x in seq:
        if x and x not in seen:
            out.append(x)
            seen.add(x)
    return out



def generate_augmented_text(query, index_name='mental', k=3, max_point_len=180):
    hits = search_documents_es(query, index_name)
    if not hits:
        return []
    hits = sorted(hits, key=lambda h: h.get("_score", 0), reverse=True)[:k]
    key_points = []
    for h in hits:
        src = h.get("_source", {})
        ctx = clean_text(src.get("Context", ""))
        rsp = clean_text(src.get("Response", ""))
        text = rsp or ctx
        if not text:
            continue
        point = _pick_sentence(text, max_len=max_point_len)
        if point:
            key_points.append(point)
    if len(key_points) >= 3:
        pass # break is removed
    key_points = _dedup(key_points)[:3]
    if not key_points:
        return []
    bullets = " • ".join(key_points)
    prompt_text = f"Background notes (do not quote verbatim): • {bullets}"
    # Changed 'from' to 'role' and 'value' to 'content'
    return [{"role": "user", "content": prompt_text}]


def trim_to_sentence(text: str) -> str:
    if not text:
        return text
    # If it already ends with ., !, or ? (optionally followed by quotes/brackets), keep as is
    if SENT_END_OK.search(text):
        return text
    # Otherwise, cut to the last sentence-ending punctuation if present
    last = None
    for m in re.finditer(r'[.!?]', text):
        last = m
    if last:
        return text[: last.end()].rstrip()
    return text


def respond(message, model, tokenizer, tokens, model_label, rag, temperature=0.7, top_p=0.9, do_sample=True):
    messages = generate_basic_messages(message) if model_label == 'basic' else generate_cot_messages(message)
    if rag:
        aug = generate_augmented_text(message)
        if aug:
            # Merge into the single user message, using 'content' key
            messages[0]["content"] = aug[0]["content"] + "\n\n" + messages[0]["content"]

    enc = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    )
    # enc can be a Tensor or a dict depending on tokenizer
    if isinstance(enc, dict):
        input_ids = enc["input_ids"].to(model.device)
        attention_mask = enc.get("attention_mask", None)
        if attention_mask is not None:
            attention_mask = attention_mask.to(model.device)
    else:
        input_ids = enc.to(model.device)
        attention_mask = None

    # Respect model context window when choosing max_new_tokens
    model_max_ctx = getattr(model.config, "max_position_embeddings", getattr(tokenizer, "model_max_length", 4096))
    input_len = input_ids.shape[-1]
    available_for_new = max(16, model_max_ctx - input_len - 1)
    max_new = min(int(tokens), available_for_new)

    gen_ids = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=max_new,
        do_sample=True,
        temperature=0.4,
        top_p=0.85,
        no_repeat_ngram_size=3,
        repetition_penalty=1.05,
        use_cache=True,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=(tokenizer.eos_token_id if tokenizer.eos_token_id is not None else tokenizer.pad_token_id),
        )

    # Decode only the newly generated tokens
    new_tokens = gen_ids[0, input_len:]
    output = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    output = trim_to_sentence(output)
    # Changed 'from' to 'role' and 'value' to 'content'
    messages.append({"role": "assistant", "content": output})
    return messages

def get_model_response(message, model_info, tokens, history, temperature=None, top_p=None):
    model_name, model_label, rag = model_info
    key = (model_name, model_label)
    history = history or []
    if key not in model_dict:
        raise ValueError("Invalid model selected.")
    model, tokenizer = model_dict[key]
    res = respond(
        message, model, tokenizer, tokens, model_label, rag == 'rag',
        temperature=(0.7 if temperature is None else temperature),
        top_p=(0.9 if top_p is None else top_p),
    )
    # Changed 'value' to 'content'
    bot_message = res[-1]["content"].replace("Assistant:", "").strip()
    history.append((message, bot_message))
    return history


def clear():
    # Clear only the message input field
    return ""

In [ ]:
message = "I'm feeling very anxious lately."
user_input = "I just found out my dog has cancer, and I'm completely devastated. I don't know how I'll cope without her. "
user_input_1 = "I just feel so down lately. I don't have the energy to do anything, and even things I used to enjoy just seem boring."
user_input_2 = "Ugh, I feel so overwhelmed lately. I have a huge presentation at work next week, and my apartment is a mess. I just can't seem to catch a break."
user_input_3 = "I am stressed, I don't know what to do, how do I resolve this feelings"
user_input_4 = "I am stressed, I don't know what to do, work has been busy, my girlfriend broke up with me"
get_model_response(user_input_2, MODEL_LIST_[3], 128, None)

[("Ugh, I feel so overwhelmed lately. I have a huge presentation at work next week, and my apartment is a mess. I just can't seem to catch a break.",
  "I know how you feel. It's hard to juggle all of our responsibilities. I would suggest taking a few minutes to just breathe and take a few deep breaths. This will help you to focus on the task at hand. I also recommend taking a break from work for a few days. You can also try to get some help from your friends or family. They can give you advice or even help you organize your apartment.")]

# EMPHABOT INTERFACE

In [ ]:
import gradio as gr

with gr.Blocks() as demo:
    gr.Markdown("## Empathetic Mental Health Chatbot Comparison")

    with gr.Row():
        # Input textbox
        user_input = gr.Textbox(label="Enter your message:", placeholder="Type a message...", show_label=False)

        # Button actions
        with gr.Column():
            submit_btn = gr.Button("Submit")
            clear_btn = gr.Button("Clear")
            restart_btn = gr.Button("Restart")
            # Token slider
            max_new_tokens_slider = gr.Slider(
                label="Maximum New Tokens",
                minimum=64,
                maximum=1024,
                value=128,
                step=64,
                interactive=True,
                visible=True,
                info="The maximum number of tokens"
            )

    with gr.Row():
        # First set of Chatbots
        chat_a = gr.Chatbot(height=500, label=f"Model {MODEL_LIST_[0]}")
        chat_b = gr.Chatbot(height=500, label=f"Model {MODEL_LIST_[1]}")

    with gr.Row():
        # Second set of Chatbots
        chat_c = gr.Chatbot(height=500, label=f"Model {MODEL_LIST_[2]}")
        chat_d = gr.Chatbot(height=500, label=f"Model {MODEL_LIST_[3]}")

    chatbots = [chat_a, chat_b, chat_c, chat_d]

    history_list = [gr.State(value=[]) for _ in MODEL_LIST_]  # Track history for each model


    # Function to get responses from all models
    def respond_to_message(message, max_new_tokens, *histories):
        outputs = []
        for i, model_info in enumerate(MODEL_LIST_):
            # Pass the model info and max_new_tokens to your get_model_response function
            output = get_model_response(message, model_info, max_new_tokens, histories[i])
            outputs.append(output)

        # Convert all histories to Gradio format
        chat_histories_for_gradio = []
        for history in outputs:
            chat_history_for_gradio = [(entry[0], entry[1]) for entry in history]
            chat_histories_for_gradio.append(chat_history_for_gradio)

        return tuple(chat_histories_for_gradio)

    submit_btn.click(
        fn=respond_to_message,
        inputs=[user_input, max_new_tokens_slider] + history_list,
        outputs=chatbots
    )

    clear_btn.click(
        clear,
        inputs=[],
        outputs=[user_input]
    )

demo.queue(default_concurrency_limit=10).launch()

/tmp/ipython-input-2391961004.py:29: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chat_a = gr.Chatbot(height=500, label=f"Model {MODEL_LIST_[0]}")
/tmp/ipython-input-2391961004.py:29: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chat_a = gr.Chatbot(height=500, label=f"Model {MODEL_LIST_[0]}")
/tmp/ipython-input-2391961004.py:30: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, wh

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0e4f820b42b31b9976.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
